# Notebook 2 — Index the elements and measure retrieval

A table stays in one chunk. Search uses the same vector-plus-keyword fusion as module 15. The score below is a retrieval score. The chat model is not asked to grade itself.

## Outline

1. Build or reuse the Chroma index.
2. Look at one chunk of each type.
3. Compare keyword search and vector search on a rare amount.
4. Score all 16 questions.

Read `elements_to_chunks` and `fuse` in `document_rag.py`. If a row below says `MISS`, open that page image before you blame the chat model.


In [1]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if not (HERE / "document_rag.py").exists():
    raise SystemExit("Set the notebook's working directory to 16_agentic_document_rag.")
sys.path.insert(0, str(HERE))

import document_rag as desk

# Reuses Chroma when the hash of parsed/elements.jsonl still matches.
# The first build embeds a few hundred chunks and takes several minutes.
chunks = desk.build_index(force=False, caption=True)
print(len(chunks), "chunks")


Index already matches the parsed files (284 chunks).
284 chunks


## How a chunk is cut

| Element | What we store | Why |
|---|---|---|
| `text` | Cut near 900 characters, on a paragraph break, with 120 characters repeated | A whole page is too much to read for one fact |
| `table` | The whole markdown grid | Splitting a table is how a system cites the wrong row |
| `figure` | The caption, whole. The page PNG stays in metadata | Search cannot see pixels. The person who signs can |

Every chunk starts with a stamp: document id, file type, page, element type. The stamp is inside the text, so both the embedder and BM25 can see it. Same idea as the handbook stamp in module 15.


In [2]:
shown = set()
for chunk in chunks:
    kind = chunk.metadata["element_type"]
    if kind in shown:
        continue
    shown.add(kind)
    print("=" * 60)
    print(chunk.metadata["chunk_id"])
    print(chunk.page_content[:450])
    print()
    if shown == {"text", "table", "figure"}:
        break


0082969-p1-text-1-1
Document 0082969 (pdf), page 1, text. – FINANCING ALTERNATIVE IN TIMES OF.
DEBT FUNDS 
– FINANCING ALTERNATIVE IN TIMES OF 
AMPLE MONETARY ACCOMMODATION 
 
 
 
 
 
Financial Research 
September 2018

1295982-p1-table-14-1
Document 1295982 (pdf), page 1, table. Portfolio Manager.
| Portfolio Manager | Hamish Douglass |
| --- | --- |
| Strategy Inception Date | 1 July 2013 |
| Total Global Equity Assets1 | USD $25,109.9 million |
| Total Strategy Assets | USD $2,726.6 million |

walker_ranch_seismic-p1-figure-70-1
Document walker_ranch_seismic (docx), page 1, figure. Walker Ranch seismic images.
Embedded figure in walker_ranch_seismic.docx. Use the paragraphs in this same file for the caption. Do not invent labels for the picture.



## Two searches, then fusion

`18,443.87` is a rare string. Keyword search should land on the Newton minutes. A vague question about "loss rate" has to lean on the vectors.

Fusion is reciprocal rank fusion, the same formula as module 15. A chunk at rank `r` adds `1 / (60 + r)`. A chunk that appears in both lists beats a chunk that appears in only one.

The `60` is a constant, not "return 60 chunks". It stops one lucky first place from drowning out a chunk both searches agree on.

`words()` keeps `18,443.87` and also stores `18443.87`, so either spelling matches.


In [3]:
def labels(docs):
    return [f"{d.metadata['doc_id']} p.{d.metadata['page']} {d.metadata['element_type']}" for d in docs]

amount = "18,443.87"
print("keyword:", labels(desk.keyword_search(amount, k=3)))
print("vector: ", labels(desk.vector_search(amount, k=3)))
print("fused:  ", labels(desk.hybrid_search(amount, k=4)))
print()
vague = "commercial real estate portfolio loss rate under the adverse scenario"
print("keyword:", labels(desk.keyword_search(vague, k=3)))
print("vector: ", labels(desk.vector_search(vague, k=3)))
print("fused:  ", labels(desk.hybrid_search(vague, k=4)))


keyword: ['2025833 p.6 text']


vector:  ['2025833 p.4 text', '2025833 p.6 text', '2025833 p.7 text']
fused:   ['2025833 p.4 text', '2025833 p.6 text', '2025833 p.6 text', '2025833 p.7 text']

keyword: ['7257900 p.3 text', '7257900 p.7 text', '7257900 p.7 figure']


vector:  ['7257900 p.7 text', '7257900 p.3 text', '7257900 p.7 figure']
fused:   ['7257900 p.7 text', '7257900 p.3 text', '7257900 p.7 figure', '7257900 p.1 text']


## The score that matters before any answer is written

`retrieval_report` runs each gold question and checks two things:

- **doc hit** — the source document is in the top 4
- **page hit** — one of the benchmark's evidence pages from that document is in the top 4

On the index built for this packet both scores were 16/16, including Walker Ranch and the PNC page whose rate lives only in the figure caption. Re-run the cell.

**How to read a miss.** A wrong answer on a missed page is a retrieval bug. A wrong answer on a hit page is a reading bug. Notebook 3's person is for the reading bug.


In [4]:
rows = desk.retrieval_report(k=4)
doc_hits = sum(row["hit_doc"] for row in rows)
page_hits = sum(row["hit_page"] for row in rows)
print(f"doc hit {doc_hits}/{len(rows)}    page hit {page_hits}/{len(rows)}\n")
for row in rows:
    mark = "ok  " if row["hit_doc"] and row["hit_page"] else "MISS"
    print(f"{mark} {row['answer_type'][:22]:22} {row['doc_id']:22} {row['top']}")


doc hit 16/16    page hit 16/16

ok   table_required         6002966                ['6002966 p.10 text', '6002966 p.5 text', '6002966 p.3 text', '6002966 p.4 text']
ok   image_plus_text_as_ans 1295982                ['1295982 p.3 text', '1295982 p.3 text', '1295982 p.3 text', '1295982 p.3 figure']
ok   table_required         6838482                ['6838482 p.7 text', '2025833 p.3 text', '2025833 p.3 text', '6838482 p.1 text']
ok   table_required         2025833                ['2025833 p.4 text', '2025833 p.6 text', '2025833 p.6 text', '2025833 p.2 text']
ok   text_only              0082969                ['0082969 p.8 text', '0082969 p.3 text', '0082969 p.9 text', '0082969 p.7 text']
ok   image_plus_text_as_ans 1295982                ['1295982 p.3 text', '1295982 p.3 figure', '1295982 p.3 text', '1295982 p.3 text']
ok   table_required         2025833                ['2025833 p.4 text', '2025833 p.3 text', '2025833 p.6 text', '2025833 p.6 text']
ok   table_required         2025833   

## Exercise

Run `hybrid_search` on these two questions and write down the top document id for each.

1. `In the Walker Ranch seismic figures, what do the red dots show?`
2. `In PNC's adverse scenario, what is the Commercial Real Estate portfolio loss rate?`

<details><summary>Answer</summary>

1. `walker_ranch_seismic`. The geothermal note is in the packet so a finance question is not the only thing the index contains. This question should not cite a shareholder letter.
2. `7257900`, and the figure chunk for page 7 should be in the list. That chunk is the checked Table 5 transcription. The prose chunk on the same page talks about CRE as a share of total losses, which is a different number.

</details>


In [5]:
for question in [
    "In the Walker Ranch seismic figures, what do the red dots show?",
    "In PNC's adverse scenario, what is the Commercial Real Estate portfolio loss rate?",
]:
    print(question)
    print(" ", labels(desk.hybrid_search(question, k=4)))
    print()


In the Walker Ranch seismic figures, what do the red dots show?
  ['walker_ranch_seismic p.1 text', 'walker_ranch_seismic p.1 text', 'walker_ranch_seismic p.1 figure', 'walker_ranch_seismic p.1 text']

In PNC's adverse scenario, what is the Commercial Real Estate portfolio loss rate?


  ['7257900 p.7 text', '7257900 p.3 text', '7257900 p.7 figure', '7257900 p.1 text']



## Pitfall

One page often becomes three or four text chunks, so the top 4 can be the same page four times. Overlap did that on purpose: the sentence with the number and the sentence with the subject can land in neighbouring chunks, and fusion may return both.

The generator still only reads those text chunks. It does not see the page image. The image is shown to the person in notebook 3. That is why a table or a figure waits for a signature even after retrieval succeeded.

Next notebook: the graph that drafts an answer and then stops for a person.
